第 1 段：安装依赖库

In [2]:
# 安装所需库（在 Jupyter 中运行一次即可）
!pip install nltk jieba spacy transformers chardet
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12

第 2 段：编码格式处理

In [6]:
# 编码转换示例
text = "示例文本".encode('gbk')  # 假设原始编码是GBK，字符串 → GBK 字节流
text = text.decode('gbk').encode('utf-8')  # 转换为UTF-8
print(text.decode('utf-8'))

# 使用 chardet 自动检测编码
import chardet
#chardet 是一个第三方库，它会统计学分析这段字节的特征（比如特定语言的字符频率、字节顺序），猜测它最可能是什么编码

raw_data = b'\xe7\xa4\xba\xe4\xbe\x8b\xe6\x96\x87\xe6\x9c\xac'
#把 raw_data 这段二进制数据（字节流）扔给 chardet 库，让它“猜”出这段数据原本是用什么字符编码（如 UTF-8、GBK、Shift-JIS 等）保存的
result = chardet.detect(raw_data)
print(f"检测到的编码: {result['encoding']}")

示例文本
检测到的编码: utf-8


第 3 段：移除 HTML 标签

In [7]:
import re
#re 是 Python 的 正则表达式模块（Regular Expression），它的核心功能是：按照某种“模式（Pattern）”去查找、匹配、替换字符串中的特定内容。
# 移除HTML标签示例
text = "<p>这是一段 HTML 文本</p>"
#把 text 字符串中所有的 HTML 标签（比如 <div>、<p>、<a href="...">）统统替换成空字符串（即直接删掉），只留下标签之间的纯文本内容。
#re.sub(模式, 替换成什么, 目标字符串)
clean_text = re.sub(r'<[^>]+>', '', text)
print(clean_text)  # 输出: 这是一段HTML文本

这是一段 HTML 文本


第 4 段：英文分词（NLTK）

In [10]:
import nltk
#导入 NLTK 库，让 Python 具备自然语言处理的能力。
nltk.download('punkt_tab')
#punkt 是 NLTK 内置的一个预训练模型数据包。它里面装着英文（及其他语言）的“断句”和“断词”规则，比如知道句号 . 在 "Mr."（先生）后面不算句子结束，但在 "Hello." 后面算结束。
from nltk.tokenize import word_tokenize

text = "Natural Language Processing is fascinating!"
tokens = word_tokenize(text)
print(tokens)
# 输出: ['Natural', 'Language', 'Processing', 'is', 'fascinating', '!']

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...


['Natural', 'Language', 'Processing', 'is', 'fascinating', '!']


[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


第 5 段：中文分词（jieba）

In [12]:
import jieba
#jieba 是 Python 中最常用的中文分词库，专门针对中文语言特点设计。
text = "自然语言处理非常有趣"
tokens = jieba.lcut(text)
print(tokens)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 1.008 seconds.
Prefix dict has been built successfully.


['自然语言', '处理', '非常', '有趣']


第 6 段：子词分词（HuggingFace Tokenizer）

In [2]:
from pathlib import Path

from transformers import BertTokenizer
# ============================================================
# 1. 指定本地文件夹
# ============================================================
# 注意：
# Windows路径前面加 r，避免 \ 被Python当成转义字符
model_dir = Path(r"D:\11\NLP\data")
# 拼接得到词表文件路径
vocab_file = model_dir / "vocab.txt"

# ============================================================
# 3. 直接通过本地 vocab.txt 创建BERT分词器
# ============================================================
#BertTokenizer 是 Hugging Face transformers 库中，专门为 BERT 提供的分词器。
tokenizer = BertTokenizer(
    vocab_file=str(vocab_file),
    do_lower_case=True,
    tokenize_chinese_chars=True
)
print("BERT中文分词器加载成功。")

# ============================================================
# 4. 测试基本分词
# ============================================================

text = "自然语言处理"
tokens = tokenizer.tokenize(text)
print("\n原始文本：")
print(text)
print("\n分词结果：")
print(tokens)

# ============================================================
# 5. 将Token转换成词表编号
# ============================================================
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("\nToken对应的编号：")
print(token_ids)

# ============================================================
# 6. 使用完整编码方法
# ============================================================
encoded = tokenizer(
    text,
    add_special_tokens=True,
    return_attention_mask=True,
    return_token_type_ids=True
)
print("\n完整编码结果：")
print(encoded)

print("\n加入特殊符号后的Token：")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

BERT中文分词器加载成功。

原始文本：
自然语言处理

分词结果：
['自', '然', '语', '言', '处', '理']

Token对应的编号：
[5632, 4197, 6427, 6241, 1905, 4415]

完整编码结果：
{'input_ids': [101, 5632, 4197, 6427, 6241, 1905, 4415, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

加入特殊符号后的Token：
['[CLS]', '自', '然', '语', '言', '处', '理', '[SEP]']


第 7 段：词性标注（spaCy）

In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Natural Language Processing is fascinating!")

for token in doc:
    print(f"{token.text}: {token.pos_}")
# 输出每个词及其词性标签
#PROPN专有名词。
# NOUN普通名词
#AUX助动词/系动词
#ADJ形容词
#PUNCT标点符号

Natural: PROPN
Language: PROPN
Processing: NOUN
is: AUX
fascinating: ADJ
!: PUNCT


补充：去除停用词（NLTK）

In [1]:
"""这段代码的核心目的是：在英文文本分析中，剔除那些“高频但无实际意义”的单词（比如 "this"、"is"、"a"、"for"），只保留能代表文本主题的关键词（比如 "sample"、"sentence"、"removing"）。这在自然语言处理（NLP）中是一个非常标准的“文本预处理”步骤，专业术语叫去除停用词（Stop Words Removal）。"""
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')

text = "This is a sample sentence for removing stop words."
tokens = word_tokenize(text.lower())
#text.lower() 先把所有字母变成小写（This 变成 this），目的是让后面的匹配不出错（因为 This 和 this 在程序里是两个字符）。
stop_words = set(stopwords.words('english'))
#获取英文停用词列表，并转换成 Python 的 set（集合）。为什么要转成集合？因为 集合（Set）的查找速度比列表（List）快几十倍，这里是为了让计算机跑得更快。
filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
#word.isalpha()：只保留由字母组成的词。这一步是为了干掉标点符号（比如末尾的 .），因为标点不是我们想分析的内容。
print("原始词:", tokens)
print("去除停用词后:", filtered_tokens)

C:\Users\Administrator\.conda\envs\rl\lib\ssl.py:570: UserWarning: unable to load Windows certificates, some may be corrupted
  warnings.warn("unable to load Windows certificates, some may be corrupted")
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...


原始词: ['this', 'is', 'a', 'sample', 'sentence', 'for', 'removing', 'stop', 'words', '.']
去除停用词后: ['sample', 'sentence', 'removing', 'stop', 'words']


[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


补充：中文去除停用词

In [2]:
import jieba

# 自定义简单中文停用词表（实际使用时建议加载完整停用词表文件）
stopwords = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好', '自己', '这'}

text = "这本书的内容非常丰富，我很喜欢阅读它。"
words = jieba.lcut(text)
filtered_words = [word for word in words if word not in stopwords and len(word.strip()) > 1]

print("原始分词:", words)
print("去除停用词后:", filtered_words)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.662 seconds.
Prefix dict has been built successfully.


原始分词: ['这', '本书', '的', '内容', '非常', '丰富', '，', '我', '很', '喜欢', '阅读', '它', '。']
去除停用词后: ['本书', '内容', '非常', '丰富', '喜欢', '阅读']


补充：完整的文本预处理流水线

In [3]:
import re
import jieba
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

def preprocess_chinese(text, remove_stopwords=True):
    """
    中文文本预处理流水线
    """
    # 1. 去除HTML标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 去除特殊字符（保留中文、英文、数字）
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)

    # 3. 分词
    words = jieba.lcut(text)

    # 4. 去除停用词（可选）
    if remove_stopwords:
        stop_words = set(stopwords.words('chinese'))  # NLTK中文停用词较少，建议补充
        # 补充自定义停用词
        extra_stopwords = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一'}
        stop_words.update(extra_stopwords)
        words = [w for w in words if w not in stop_words and len(w.strip()) > 1]

    return words

# 测试
sample = "<p>我喜欢学习自然语言处理，这本书真的很有趣！</p>"
result = preprocess_chinese(sample)
print(result)

['喜欢', '学习', '自然语言', '本书', '真的', '有趣']
